# LlamaCppLM Demo

CPU inference with GGUF models via `llama-cpp-python`, using the same
`LM[int]` / `LMState[int]` interface as `HuggingFaceLM`.

In [1]:
from transduction.lm.llama_cpp_lm import LlamaCppLM
from huggingface_hub import hf_hub_download
import time
import numpy as np

## Load model

Download a GPT-2 GGUF (Q8_0 ≈ nearly lossless) and wrap it.

In [2]:
gguf_path = hf_hub_download(repo_id='QuantFactory/gpt2-GGUF', filename='gpt2.Q8_0.gguf')
lm = LlamaCppLM.from_file(gguf_path, n_ctx=512)
print(f'Vocab size: {lm._n_vocab}')
print(f'EOS token:  {lm.eos}')

llama_context: n_ctx_per_seq (512) < n_ctx_train (1024) -- the full capacity of the model will not be utilized
llama_kv_cache_unified: LLAMA_SET_ROWS=0, using old ggml_cpy() method for backwards compatibility


Vocab size: 50257
EOS token:  50256


## Basic usage

In [ ]:
s = lm.initial()
s.logp_next.top(5)

In [ ]:
# Condition on a prefix
s2 = s >> lm._encode[b' The'] >> lm._encode[b' U'] >> lm._encode[b'.'] >> lm._encode[b'S'] >> lm._encode[b'.']
s2.logp_next.top(5)

In [ ]:
# Tree branching — two children, parent unaffected
s_a = s >> lm._encode[b' a']
s_b = s >> lm._encode[b' b']
print('After " a":', list(s_a.logp_next.top(3).items()))
print('After " b":', list(s_b.logp_next.top(3).items()))
print('Parent still:', list(s.logp_next.top(3).items()))

## Greedy decoding

In [ ]:
tokens = lm.initial().greedy_decode(max_len=40)
text = b''.join(lm._decode[t] for t in tokens)
print(text.decode('utf-8', errors='replace'))

## Sampling

In [ ]:
tokens = lm.initial().sample_decode(max_len=40)
text = b''.join(lm._decode[t] for t in tokens)
print(text.decode('utf-8', errors='replace'))

## Throughput benchmark

Measures tokens/sec for sequential greedy decoding (the typical
use case for this backend: tree-structured LM queries where each
step requires a KV-cache restore + single-token eval).

In [ ]:
def bench_greedy(lm, n_tokens=100, warmup=10):
    """Greedy-decode n_tokens, return tokens/sec."""
    # Warmup
    s = lm.initial()
    for _ in range(warmup):
        best = s.logp_next.argmax()
        s = s >> best

    # Timed run (fresh chain)
    s = lm.initial()
    t0 = time.perf_counter()
    for _ in range(n_tokens):
        best = s.logp_next.argmax()
        if best == lm.eos:
            break
        s = s >> best
    elapsed = time.perf_counter() - t0
    return n_tokens / elapsed, elapsed

tok_sec, elapsed = bench_greedy(lm, n_tokens=100)
print(f'{tok_sec:.1f} tokens/sec  ({elapsed:.2f}s for 100 tokens)')

In [ ]:
def bench_branching(lm, depth=20, fan_out=5):
    """Simulate tree-structured queries: at each depth, branch into
    fan_out children, keep only the best, repeat.  This exercises
    KV-cache save/restore on every step.
    """
    s = lm.initial()
    top_tids = sorted(s.logp_next.top_ids(fan_out).keys())  # warmup initial

    n_evals = 0
    t0 = time.perf_counter()
    for _ in range(depth):
        top_tids = list(s.logp_next.top_ids(fan_out).keys())
        children = [s >> tid for tid in top_tids]
        # Force eval of all children
        for c in children:
            _ = c.logp_next
        n_evals += fan_out
        # Keep best child
        s = max(children, key=lambda c: c.logp)
    elapsed = time.perf_counter() - t0
    return n_evals / elapsed, n_evals, elapsed

tok_sec, n_evals, elapsed = bench_branching(lm, depth=20, fan_out=5)
print(f'{tok_sec:.1f} evals/sec  ({n_evals} evals in {elapsed:.2f}s)')
print(f'(each eval = restore parent KV cache + eval 1 token + save KV cache)')